# Beyond Vector RAG: Agentic, Graph and Vectorless Retrieval

The full code of [Beyond Vector RAG: Agentic, Graph and Vectorless Retrieval](https://residual-stream.lvtan71.workers.dev/posts/agentic-graph-and-vectorless-rag/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/agentic-graph-and-vectorless-rag) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/agentic-graph-and-vectorless-rag.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 10 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 10 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.18.*" huggingface_hub sentence-transformers

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "agentic-graph-and-vectorless-rag"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/agentic-graph-and-vectorless-rag


In [2]:
!tar xzf blog-b602ebd.tar.gz

## `graphrag_cost.py`

What a GraphRAG index of this blog would cost, against a vector index: the corpus in tokens and chunks, the LLM calls and tokens of indexing at the paper's ratios, and the tokens one global-search query reads.

In [3]:
!{sys.executable} graphrag_cost.py

corpus: 557,546 tokens -> 1,115 chunks of 600 tokens
vector index: 1,115 chunks embedded once, 0 LLM calls

GraphRAG at 1.42 summaries per chunk: 3,345 extraction calls + 1,583 summaries = 4,928 LLM calls
  14.5M tokens in, 2.2M out (26x the corpus as input); $2.34 at $0.10/$0.40 per M, $76.78 at $3/$15 per M
GraphRAG at 1.61 summaries per chunk: 3,345 extraction calls + 1,795 summaries = 5,140 LLM calls
  15.3M tokens in, 2.4M out (27x the corpus as input); $2.48 at $0.10/$0.40 per M, $81.55 at $3/$15 per M

per global-search query: 16,725 tokens of summaries at C0, 468,300 at C3 (map calls of up to 8,000 tokens each, before prompts: 3 and 59)
per vector-search query: one embedding, then the top 5 chunks, about 3,000 tokens


The post printed:

```text
corpus: 557,546 tokens -> 1,115 chunks of 600 tokens
vector index: 1,115 chunks embedded once, 0 LLM calls

GraphRAG at 1.42 summaries per chunk: 3,345 extraction calls + 1,583 summaries = 4,928 LLM calls
  14.5M tokens in, 2.2M out (26x the corpus as input); $2.34 at $0.10/$0.40 per M, $76.78 at $3/$15 per M
GraphRAG at 1.61 summaries per chunk: 3,345 extraction calls + 1,795 summaries = 5,140 LLM calls
  15.3M tokens in, 2.4M out (27x the corpus as input); $2.48 at $0.10/$0.40 per M, $81.55 at $3/$15 per M

per global-search query: 16,725 tokens of summaries at C0, 468,300 at C3 (map calls of up to 8,000 tokens each, before prompts: 3 and 59)
per vector-search query: one embedding, then the top 5 chunks, about 3,000 tokens
```

## `compare.py`

30 hand-written questions about the blog, each answered by one known section: BM25, dense (MiniLM), hybrid, and vectorless navigation by Qwen2.5-3B-Instruct in three variants and two modes, plus the reversed-list and section-only checks.

In [4]:
!{sys.executable} compare.py

446 sections in 66 posts; 30 questions; counts are out of 30

method                                     right post  right section  section in top 5


keyword (BM25)                                     17             12                20


dense (MiniLM)                                     14             10                20


hybrid (fused)                                     20             13                24


vectorless, titles, number only                     2              1                 -
   59 calls, 1,374 prompt tokens per question, 1 invalid replies; picks by fifth of the list 2/0/7/2/18


...

vectorless, +descriptions, reasoning                0              0                 -
   60 calls, 8,091 prompt tokens per question, 5 invalid replies


vectorless, 3 levels, reasoning                     3              2                 -
   79 calls, 2,122 prompt tokens per question, 14 invalid replies; right topic 6 (of 13 topics), then right post 3 of those


   titles reversed: right post 1; picks by fifth of the reversed list 14/3/10/0/3


   section only, right post given: 20 (chance 4.1); 1 invalid replies



questions where BM25 found the right post and dense search didn't:
   'What limits the speed of the plain attention implementation on': rarest shared words ['implementation', 'speed', 'gpu']
   'How does the preference loss follow from maximizing reward und': rarest shared words ['preference', 'kl', 'reward']
   "How is the output kept exactly distributed as the large model'": rarest shared words ['guesser', 'distributed', 'exactly']
   'How do two sequences share the same prompt memory until one of': rarest shared words ['share', 'sequences', 'prompt']
   "What maps a sequence's logical positions to physical memory ch": rarest shared words ['maps', 'logical', 'physical']
   'How is term frequency saturated and document length normalized': rarest shared words ['saturated', 'keyword', 'document']


The post printed:

```text
446 sections in 66 posts; 30 questions; counts are out of 30

method                                     right post  right section  section in top 5
keyword (BM25)                                     17             12                20
dense (MiniLM)                                     14             10                20
hybrid (fused)                                     20             12                24
vectorless, titles, number only                     2              1                 -
   59 calls, 1,374 prompt tokens per question, 1 invalid replies; picks by fifth of the list 2/0/7/2/18
vectorless, +descriptions, number only              0              0                 -
   60 calls, 8,084 prompt tokens per question, 3 invalid replies
vectorless, 3 levels, number only                   3              3                 -
   88 calls, 2,467 prompt tokens per question, 1 invalid replies; right topic 12 (of 13 topics), then right post 3 of those
   titles reversed: right post 0; picks by fifth of the reversed list 5/0/20/0/5
   section only, right post given: 18 (chance 4.1); 0 invalid replies
vectorless, titles, reasoning                       0              0                 -
   60 calls, 1,406 prompt tokens per question, 7 invalid replies; picks by fifth of the list 5/4/11/0/10
vectorless, +descriptions, reasoning                0              0                 -
   60 calls, 8,091 prompt tokens per question, 5 invalid replies
vectorless, 3 levels, reasoning                     3              2                 -
   79 calls, 2,146 prompt tokens per question, 15 invalid replies; right topic 6 (of 13 topics), then right post 3 of those
   titles reversed: right post 1; picks by fifth of the reversed list 14/3/10/0/3
   section only, right post given: 20 (chance 4.1); 1 invalid replies

questions where BM25 found the right post and dense search didn't:
   'What limits the speed of the plain attention implementation on': rarest shared words ['implementation', 'speed', 'gpu']
   'How does the preference loss follow from maximizing reward und': rarest shared words ['preference', 'kl', 'reward']
   "How is the output kept exactly distributed as the large model'": rarest shared words ['guesser', 'distributed', 'exactly']
   'How do two sequences share the same prompt memory until one of': rarest shared words ['share', 'sequences', 'prompt']
   "What maps a sequence's logical positions to physical memory ch": rarest shared words ['maps', 'logical', 'physical']
   'How is term frequency saturated and document length normalized': rarest shared words ['saturated', 'keyword', 'document']
```